# 2.3. Подготовка и организация данных

Ноутбук готовит источники, полученные на этапе 2.2: каталоги Steam и RAWG, Steam Recommendations, Steam Monthly Players, Twitch, YouTube, Habr и Wikidata.

Исходные файлы остаются в `raw/`. Подготовленные таблицы сохраняются в Parquet, результаты NLP — отдельно, аналитические таблицы и связи — в DuckDB. Повторный запуск пересоздаёт результаты из исходных данных.

Steam `games.csv` и `games.json`, а также CSV, JSONL и Parquet RAWG — повторные представления соответствующих источников. Для подготовки используются Steam JSON и RAWG Parquet. Остальные представления остаются в исходном слое и учитываются в инвентаризации.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import re
import time
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import ijson

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "notebooks").is_dir() and (p / "README.md").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Запустите ноутбук из папки проекта или notebooks/."
    )

RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"
GAMES = PROCESSED / "games"
RECOMMENDATIONS = PROCESSED / "recommendations"
PLAYERS = PROCESSED / "players"
TWITCH = PROCESSED / "twitch"
YOUTUBE = PROCESSED / "youtube"
TEXT = PROCESSED / "text"
WAREHOUSE = ROOT / "warehouse"
QUALITY_DIR = ROOT / "docs" / "data_quality"

for directory in [
    GAMES, RECOMMENDATIONS, PLAYERS, TWITCH,
    YOUTUBE, TEXT, WAREHOUSE, QUALITY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def locate(relative_path):
    path = RAW / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла из 2.2: {path}")
    return path

INPUTS = {
    "steam_games_json": locate("steam/fronkongames/games.json"),
    "rawg_parquet": locate("rawg/videogames_data.parquet"),
    "recommendation_games": locate("steam/recommendations/games.csv"),
    "recommendations": locate("steam/recommendations/recommendations.csv"),
    "steam_users": locate("steam/recommendations/users.csv"),
    "game_metadata": locate("steam/recommendations/games_metadata.json"),
    "monthly_players": locate("steam/monthly_players/steamcharts.csv"),
    "twitch_games": locate("twitch/evolution/Twitch_game_data.csv"),
    "twitch_global": locate("twitch/evolution/Twitch_global_data.csv"),
    "twitch_streamers": locate("twitch/streamers/datasetV2.csv"),
    "youtube": locate("youtube/youtube_videos.csv"),
    "habr": locate("habr/articles.json"),
    "wikidata": locate("xml/wikidata/wikidata_video_games.xml"),
}

db = duckdb.connect(":memory:")
stage_times = []
stage_start = time.perf_counter()

print("Корень проекта:", ROOT)
print("Источниковых файлов для подготовки:", len(INPUTS))

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Источниковых файлов для подготовки: 13


## Аналитический смысл данных

| Источник | Единица наблюдения и ключ | Основные атрибуты: смысл | Шкала и роль |
|---|---|---|---|
| Steam Games | Игра; `AppID` | Дата выпуска, цена, жанры, категории, положительные и отрицательные оценки, рекомендации | ID и категории — номинальные; дата — интервальная; цена и счётчики — отношений. Характеристики игры и показатели интереса |
| RAWG | Игра; `id` | Дата выпуска, рейтинг, число оценок и отзывов, жанры, платформы, разработчики | ID и категории — номинальные; рейтинг — интервальная; счётчики — отношений. Характеристики и популярность |
| Steam Recommendations | Рекомендация; `review_id`; игра — `app_id`; автор — `user_id` | Рекомендована ли игра, дата, наигранные часы, helpful/funny | ID — номинальные; дата — интервальная; часы и реакции — отношений; рекомендация — бинарная. Отношение пользователей к игре |
| Steam Recommendation Games | Игра; `app_id` | Название, цена, рейтинг и доля положительных отзывов | ID и категории — номинальные; цена и счётчики — отношений; доля — относительная. Характеристики и интерес |
| Steam Users | Пользователь; `user_id` | Число продуктов и отзывов пользователя | ID — номинальная; счётчики — отношений. Активность пользователя |
| Steam Monthly Players | Игра за месяц; `steam_appid` + `month` | Среднее/пиковое число игроков и изменение к предыдущему месяцу | ID — номинальный; месяц — временная ось; игроки — отношений. Динамика популярности |
| Twitch Games | Игра за месяц; название + год + месяц | Часы просмотра и трансляций, зрители, каналы, стримеры | Название — номинальное; год/месяц — временная ось; счётчики — отношений. Зрительский интерес и предложение контента |
| Twitch Global | Платформа за месяц; год + месяц | Общие просмотры, зрители, каналы и число транслируемых игр | Время — временная ось; счётчики — отношений. Общая динамика платформы |
| Twitch Streamers | Стример; `NAME` | Язык, наиболее транслируемые игры, подписчики, просмотры и активность | Имя/язык/игра — номинальные; счётчики — отношений. Аудитория и активность |
| YouTube | Видео; `video_id` | Заголовок, описание, поисковый запрос, просмотры, лайки и комментарии | ID — номинальный; дата — временная ось; счётчики — отношений. Интерес к игровому видеоконтенту |
| Habr | Статья; URL | Заголовок и текст статьи | URL — номинальный; текст — неструктурированный. Тематический контекст индустрии |
| Wikidata | Строка результата SPARQL; `game` | Название, дата выпуска, жанр, платформа, разработчик, издатель | ID/категории — номинальные; дата — временная ось. Дополнительные характеристики |

Идентификаторы Steam, RAWG и Wikidata принадлежат разным системам. Совпадение названия — только кандидат на связь, а не доказательство идентичности. `game_name` YouTube — поисковый запрос, не подтверждённая принадлежность ролика к игре.

Для проверки гипотез о популярности основными показателями будут активность игроков Steam и Steam Monthly Players, оценки Steam/RAWG и зрительские показатели Twitch/YouTube. Текст Habr используется как качественный контекст, а не как репрезентативная выборка индустрии.

## Инвентаризация фактически полученных файлов

In [2]:
inventory_rows = []

for path in sorted(RAW.rglob("*")):
    if path.is_file():
        inventory_rows.append({
            "relative_path": path.relative_to(ROOT).as_posix(),
            "extension": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / 1024**2, 2),
        })

inventory = pd.DataFrame(inventory_rows)
inventory_path = QUALITY_DIR / "raw_inventory_2_3.csv"
inventory.to_csv(inventory_path, index=False, encoding="utf-8-sig")

display(inventory)
print("Исходных файлов:", len(inventory))
print("Общий объём, GB:", round(inventory["size_mb"].sum() / 1024, 2))

stage_times.append({
    "stage": "raw_inventory",
    "seconds": time.perf_counter() - stage_start,
})
stage_start = time.perf_counter()

,relative_path,extension,size_mb
0,raw/habr/articles.json,.json,0.03
1,raw/rawg/jsonl/videogames_data.jsonl,.jsonl,4101.47
2,raw/rawg/videogames_data.csv,.csv,1907.80
3,raw/rawg/videogames_data.parquet,.parquet,964.80
4,raw/steam/fronkongames/games.csv,.csv,382.56
5,raw/steam/fronkongames/games.json,.json,973.38
6,raw/steam/monthly_players/steamcharts.csv,.csv,36.13
7,raw/steam/recommendations/games.csv,.csv,4.64
8,raw/steam/recommendations/games_metadata.json,.json,16.74
9,raw/steam/recommendations/recommendations.csv,.csv,1929.50


Исходных файлов: 20
Общий объём, GB: 10.28


## Steam Games и RAWG

In [3]:
def sql_literal(path):
    return "'" + Path(path).resolve().as_posix().replace("'", "''") + "'"

def to_text(value):
    return None if value is None else str(value)

def to_float(value):
    try:
        number = pd.to_numeric(value, errors="coerce")
        return None if pd.isna(number) else float(number)
    except (TypeError, ValueError):
        return None

def to_int(value):
    number = to_float(value)
    if number is None or not math.isfinite(number) or not number.is_integer():
        return None
    return int(number)

def to_json_text(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False)

STEAM_JSON = RAW / "steam" / "fronkongames" / "games.json"
steam_output = GAMES / "steam_games.parquet"

steam_schema = pa.schema([
    ("AppID", pa.string()),
    ("name", pa.string()),
    ("release_date", pa.string()),
    ("price", pa.float64()),
    ("genres", pa.string()),
    ("categories", pa.string()),
    ("positive", pa.int64()),
    ("negative", pa.int64()),
    ("recommendations", pa.int64()),
])

steam_fields = [
    "AppID", "name", "release_date", "price",
    "genres", "categories", "positive",
    "negative", "recommendations",
]

writer = None
batch = []
steam_rows = 0

try:
    with open(INPUTS["steam_games_json"], "rb") as source:
        for app_id, game in ijson.kvitems(source, ""):
            batch.append({
                "AppID": str(app_id),
                "name": to_text(game.get("name")),
                "release_date": to_text(game.get("release_date")),
                "price": to_float(game.get("price")),
                "genres": to_json_text(game.get("genres")),
                "categories": to_json_text(game.get("categories")),
                "positive": to_int(game.get("positive")),
                "negative": to_int(game.get("negative")),
                "recommendations": to_int(game.get("recommendations")),
            })

            if len(batch) >= 10_000:
                table = pa.Table.from_pylist(batch, schema=steam_schema)
                if writer is None:
                    writer = pq.ParquetWriter(
                        steam_output, steam_schema, compression="zstd"
                    )
                writer.write_table(table)
                steam_rows += len(batch)
                batch.clear()

    if batch:
        table = pa.Table.from_pylist(batch, schema=steam_schema)
        if writer is None:
            writer = pq.ParquetWriter(
                steam_output, steam_schema, compression="zstd"
            )
        writer.write_table(table)
        steam_rows += len(batch)
finally:
    if writer is not None:
        writer.close()

if steam_rows == 0:
    raise ValueError("В Steam JSON не найдено записей.")

rawg_output = GAMES / "rawg_games.parquet"
rawg_file = pq.ParquetFile(INPUTS["rawg_parquet"])

rawg_columns = [
    "id", "name", "released", "rating", "rating_top",
    "ratings_count", "reviews_count", "metacritic", "playtime",
    "genres", "platforms", "developers", "publishers",
]
available = set(rawg_file.schema_arrow.names)
missing = {"id", "name"} - available
if missing:
    raise ValueError(f"В RAWG отсутствуют обязательные поля: {missing}")

rawg_columns = [c for c in rawg_columns if c in available]
rawg_schema = pa.schema([
    rawg_file.schema_arrow.field(column)
    for column in rawg_columns
])

rawg_writer = None
rawg_rows = 0

try:
    for record_batch in rawg_file.iter_batches(
        batch_size=50_000,
        columns=rawg_columns,
    ):
        if rawg_writer is None:
            rawg_writer = pq.ParquetWriter(
                rawg_output, rawg_schema, compression="zstd"
            )
        rawg_writer.write_batch(record_batch)
        rawg_rows += record_batch.num_rows
finally:
    if rawg_writer is not None:
        rawg_writer.close()

print("Steam Games:", steam_rows, "строк")
print("RAWG:", rawg_rows, "строк")
print("Поля RAWG:", rawg_columns)

stage_times.append({
    "stage": "prepare_steam_rawg",
    "seconds": time.perf_counter() - stage_start,
})
stage_start = time.perf_counter()

Steam Games: 150466 строк
RAWG: 899585 строк
Поля RAWG: ['id', 'name', 'released', 'rating', 'rating_top', 'ratings_count', 'reviews_count', 'metacritic', 'playtime', 'genres', 'platforms', 'developers', 'publishers']


## Табличные источники

In [4]:
def csv_to_parquet(source, destination, sample_size=20_000):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.unlink(missing_ok=True)

    source_sql = sql_literal(source)
    output_sql = sql_literal(destination)

    db.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                {source_sql},
                header = true,
                sample_size = {sample_size}
            )
        )
        TO {output_sql}
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    return destination

prepared = {
    "steam_games": steam_output,
    "rawg_games": rawg_output,
}

csv_sources = {
    "steam_recommendation_games": (
        INPUTS["recommendation_games"],
        RECOMMENDATIONS / "steam_recommendation_games.parquet",
    ),
    "steam_recommendations": (
        INPUTS["recommendations"],
        RECOMMENDATIONS / "steam_recommendations.parquet",
    ),
    "steam_users": (
        INPUTS["steam_users"],
        RECOMMENDATIONS / "steam_users.parquet",
    ),
    "steam_monthly_players": (
        INPUTS["monthly_players"],
        PLAYERS / "steam_monthly_players.parquet",
    ),
    "youtube_search_rows": (
        INPUTS["youtube"],
        YOUTUBE / "youtube_search_rows.parquet",
    ),
}

for table_name, (source, destination) in csv_sources.items():
    prepared[table_name] = csv_to_parquet(source, destination)

# Twitch файлы небольшие; для игрового набора учитываем кодировку из 2.3.
twitch_csvs = {
    "twitch_games": (INPUTS["twitch_games"], "cp1252"),
    "twitch_global": (INPUTS["twitch_global"], "utf-8"),
    "twitch_streamers": (INPUTS["twitch_streamers"], "utf-8"),
}

for table_name, (source, encoding) in twitch_csvs.items():
    destination = TWITCH / f"{table_name}.parquet"
    destination.unlink(missing_ok=True)
    frame = pd.read_csv(source, encoding=encoding)
    frame.to_parquet(destination, index=False, compression="zstd")
    prepared[table_name] = destination

# Метаданные Steam Recommendations хранятся в формате JSON Lines.
metadata_df = pd.read_json(
    INPUTS["game_metadata"],
    lines=True,
    encoding="utf-8",
)

metadata_output = RECOMMENDATIONS / "steam_game_metadata.parquet"
metadata_output.unlink(missing_ok=True)

metadata_df.to_parquet(
    metadata_output,
    index=False,
    compression="zstd",
)

prepared["steam_game_metadata"] = metadata_output

print("Метаданные Steam Recommendations:", len(metadata_df), "строк")
print("Поля:", metadata_df.columns.tolist())

print("Подготовлено таблиц:", len(prepared))
display(pd.DataFrame([
    {"table": name, "file": path.relative_to(ROOT).as_posix()}
    for name, path in prepared.items()
]))

stage_times.append({
    "stage": "prepare_csv_twitch_youtube_metadata",
    "seconds": time.perf_counter() - stage_start,
})
stage_start = time.perf_counter()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Метаданные Steam Recommendations: 50872 строк
Поля: ['app_id', 'description', 'tags']
Подготовлено таблиц: 11


,table,file
0,steam_games,processed/games/steam_games.parquet
1,rawg_games,processed/games/rawg_games.parquet
2,steam_recommendation_games,processed/recommendations/steam_recommendation...
3,steam_recommendations,processed/recommendations/steam_recommendation...
4,steam_users,processed/recommendations/steam_users.parquet
5,steam_monthly_players,processed/players/steam_monthly_players.parquet
6,youtube_search_rows,processed/youtube/youtube_search_rows.parquet
7,twitch_games,processed/twitch/twitch_games.parquet
8,twitch_global,processed/twitch/twitch_global.parquet
9,twitch_streamers,processed/twitch/twitch_streamers.parquet


## Приводим данные Steam Monthly Players к определенному формату

In [5]:
monthly_path = prepared["steam_monthly_players"]
monthly = pd.read_parquet(monthly_path)

monthly["month_raw"] = monthly["month"].astype("string")
monthly["month"] = pd.to_datetime(
    monthly["month_raw"],
    format="%b-%y",
    errors="coerce",
)

monthly["gain_raw"] = monthly["gain"].astype("string")
monthly["gain"] = pd.to_numeric(
    monthly["gain_raw"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False),
    errors="coerce",
)

monthly.to_parquet(
    monthly_path,
    index=False,
    compression="zstd",
)

print("Не удалось разобрать месяц:", int(monthly["month"].isna().sum()))
print("Не удалось преобразовать gain:", int(monthly["gain"].isna().sum()))
display(monthly[["month", "month_raw", "gain", "gain_raw"]].head())

Не удалось разобрать месяц: 0
Не удалось преобразовать gain: 6728


,month,month_raw,gain,gain_raw
0,2025-09-01,Sep-25,883.12,883.12
1,2025-08-01,Aug-25,-449.35,-449.35
2,2025-07-01,Jul-25,-833.5,-833.5
3,2025-06-01,Jun-25,-847.53,-847.53
4,2025-05-01,May-25,-471.31,-471.31


## Wikidata и Habr

In [6]:
# Wikidata XML -> табличные строки.
namespace = {"ns": "http://www.w3.org/2005/sparql-results#"}
wikidata_records = []

for event, element in ET.iterparse(
    INPUTS["wikidata"], events=("end",)
):
    if element.tag.endswith("result"):
        record = {}
        for binding in element.findall("ns:binding", namespace):
            key = binding.attrib["name"]
            value = binding.find("ns:uri", namespace)
            if value is None:
                value = binding.find("ns:literal", namespace)
            record[key] = value.text if value is not None else None

        wikidata_records.append(record)
        element.clear()

wikidata_df = pd.DataFrame(wikidata_records)
wikidata_output = GAMES / "wikidata_games.parquet"
wikidata_output.unlink(missing_ok=True)
wikidata_df.to_parquet(
    wikidata_output, index=False, compression="zstd"
)
prepared["wikidata_games"] = wikidata_output

# Habr: исходный текст сохраняется; очищенная версия нужна для NLP.
with open(INPUTS["habr"], "r", encoding="utf-8") as file:
    habr_df = pd.DataFrame(json.load(file))

def clean_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value)).lower()
    value = re.sub(r"https?://\S+", " ", value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"[^a-zа-яё0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

habr_df["clean_text"] = (
    habr_df.get("title", "").fillna("").astype(str)
    + " "
    + habr_df.get("text", "").fillna("").astype(str)
).map(clean_text)

habr_output = TEXT / "habr_articles.parquet"
habr_output.unlink(missing_ok=True)
habr_df.to_parquet(habr_output, index=False, compression="zstd")
prepared["habr_articles"] = habr_output

print("Wikidata:", len(wikidata_df), "строк")
print("Habr:", len(habr_df), "статей")

stage_times.append({
    "stage": "prepare_wikidata_habr",
    "seconds": time.perf_counter() - stage_start,
})
stage_start = time.perf_counter()

Wikidata: 1000 строк
Habr: 2 статей


## Профиль качества

Для каждой подготовленной таблицы проверяются типы, пропуски, дубликаты ключей и диапазоны числовых полей. Первые строки служат примером структуры данных.

Повторные представления Steam Games и RAWG остаются в `raw/` и не загружаются повторно в `processed/`. Для больших наборов сводная проверка выполняется DuckDB, без полной загрузки данных в pandas.

In [7]:
key_columns = {
    "steam_games": ["AppID"],
    "rawg_games": ["id"],
    "steam_recommendation_games": ["app_id"],
    "steam_recommendations": ["review_id"],
    "steam_users": ["user_id"],
    "steam_monthly_players": ["steam_appid", "month"],
    "youtube_search_rows": ["video_id", "game_name"],
    "twitch_games": ["Game", "Year", "Month"],
    "twitch_global": ["year", "Month"],
    "twitch_streamers": ["NAME"],
    "steam_game_metadata": ["app_id"],
    "wikidata_games": [
        "game", "releaseDate", "genreLabel", "platformLabel",
        "developerLabel", "publisherLabel",
    ],
    "habr_articles": ["url"],
}

def quote_identifier(value):
    return '"' + value.replace('"', '""') + '"'

quality_rows = []
summary_rows = []
range_rows = []

for table_name, path in prepared.items():
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    columns = schema.names
    row_count = parquet.metadata.num_rows
    parquet_sql = sql_literal(path)

    expressions = ["COUNT(*) AS row_count"]
    missing_aliases = {}
    range_aliases = {}

    for index, column in enumerate(columns):
        quoted = quote_identifier(column)
        field_type = schema.field(column).type
        missing_alias = f"missing_{index}"

        if pa.types.is_string(field_type) or pa.types.is_large_string(field_type):
            missing_condition = f"({quoted} IS NULL OR TRIM({quoted}) = '')"
        else:
            missing_condition = f"({quoted} IS NULL)"

        expressions.append(
            f"SUM(CASE WHEN {missing_condition} THEN 1 ELSE 0 END) "
            f"AS {missing_alias}"
        )
        missing_aliases[column] = missing_alias

        if (
            pa.types.is_integer(field_type)
            or pa.types.is_floating(field_type)
            or pa.types.is_decimal(field_type)
        ):
            min_alias = f"min_{index}"
            max_alias = f"max_{index}"
            expressions.extend([
                f"MIN({quoted}) AS {min_alias}",
                f"MAX({quoted}) AS {max_alias}",
            ])
            range_aliases[column] = (min_alias, max_alias)

    result = db.execute(
        f"SELECT {', '.join(expressions)} "
        f"FROM read_parquet({parquet_sql})"
    ).fetchone()

    result_by_alias = {
        alias: result[position]
        for position, alias in enumerate(
            ["row_count"]
            + [
                alias
                for column in columns
                for alias in (
                    [missing_aliases[column]]
                    + list(range_aliases.get(column, ()))
                )
            ],
            start=0,
        )
    }

    for column in columns:
        missing_count = int(result_by_alias[missing_aliases[column]] or 0)
        quality_rows.append({
            "table": table_name,
            "column": column,
            "data_type": str(schema.field(column).type),
            "missing_count": missing_count,
            "missing_percent": round(
                100 * missing_count / row_count, 2
            ) if row_count else 0,
        })

        if column in range_aliases:
            min_alias, max_alias = range_aliases[column]
            range_rows.append({
                "table": table_name,
                "column": column,
                "minimum": result_by_alias[min_alias],
                "maximum": result_by_alias[max_alias],
            })

    keys = key_columns.get(table_name, [])
    duplicate_key_rows = None

    if keys:
        quoted_keys = ", ".join(quote_identifier(key) for key in keys)
        duplicate_key_rows = db.execute(f"""
            SELECT COALESCE(SUM(group_rows - 1), 0)
            FROM (
                SELECT COUNT(*) AS group_rows
                FROM read_parquet({parquet_sql})
                GROUP BY {quoted_keys}
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

    summary_rows.append({
        "table": table_name,
        "rows": row_count,
        "columns": len(columns),
        "duplicate_key_rows": int(duplicate_key_rows or 0)
        if duplicate_key_rows is not None else None,
    })

    sample_columns = columns[:8]
    sample_batch = next(
        parquet.iter_batches(batch_size=3, columns=sample_columns),
        None,
    )

    print(f"\n{table_name}: {row_count:,} строк, {len(columns)} полей")
    display(pd.DataFrame({
        "column": columns,
        "type": [str(schema.field(column).type) for column in columns],
    }))

    if sample_batch is not None:
        display(sample_batch.to_pandas())

quality_df = pd.DataFrame(quality_rows)
summary_df = pd.DataFrame(summary_rows)
ranges_df = pd.DataFrame(range_rows)

quality_df.to_csv(
    QUALITY_DIR / "column_quality_2_3.csv",
    index=False,
    encoding="utf-8-sig",
)
summary_df.to_csv(
    QUALITY_DIR / "table_summary_2_3.csv",
    index=False,
    encoding="utf-8-sig",
)
ranges_df.to_csv(
    QUALITY_DIR / "numeric_ranges_2_3.csv",
    index=False,
    encoding="utf-8-sig",
)

display(summary_df)
display(ranges_df)


steam_games: 150,466 строк, 9 полей


,column,type
0,AppID,string
1,name,string
2,release_date,string
3,price,double
4,genres,string
5,categories,string
6,positive,int64
7,negative,int64
8,recommendations,int64


,AppID,name,release_date,price,genres,categories,positive,negative
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.00,[],[],0,0
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,"[""Adventure""]","[""Single-player"", ""Steam Trading Cards"", ""Stea...",252,3
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"[""Casual""]","[""Single-player"", ""Family Sharing""]",21,3



rawg_games: 899,585 строк, 13 полей


,column,type
0,id,int64
1,name,string
2,released,string
3,rating,double
4,rating_top,double
5,ratings_count,double
6,reviews_count,double
7,metacritic,double
8,playtime,double
9,genres,string


,id,name,released,rating,rating_top,ratings_count,reviews_count,metacritic
0,3498,Grand Theft Auto V,2013-09-17,4.47,5.0,7409.0,7544.0,92.0
1,3328,The Witcher 3: Wild Hunt,2015-05-18,4.64,5.0,7234.0,7354.0,92.0
2,4200,Portal 2,2011-04-18,4.58,5.0,6118.0,6175.0,95.0


... (31 outputs left out)

,column,type
0,url,large_string
1,title,large_string
2,status_code,int64
3,text,large_string
4,clean_text,large_string


,url,title,status_code,text,clean_text
0,https://habr.com/ru/articles/995396/,"Только 12,5% игр в Steam получают пользователь...",200,Data-driven разбор рынка Steam 2021–2025 Кажды...,только 12 5 игр в steam получают пользовательс...
1,https://habr.com/ru/articles/990554/,От ниши к мейнстриму: с 33 до 75% за 5 лет. Ка...,200,В начале 2022 года РВИ вместе с НАФИ сделали «...,от ниши к мейнстриму с 33 до 75 за 5 лет как и...


,table,rows,columns,duplicate_key_rows
0,steam_games,150466,9,0
1,rawg_games,899585,13,0
2,steam_recommendation_games,50872,13,0
3,steam_recommendations,41154794,8,0
4,steam_users,14306064,3,0
5,steam_monthly_players,612265,9,0
6,youtube_search_rows,299,9,0
7,twitch_games,21000,12,10
8,twitch_global,105,9,0
9,twitch_streamers,999,17,0


,table,column,minimum,maximum
0,steam_games,price,0.0,9.999900e+02
1,steam_games,positive,0.0,7.642084e+06
2,steam_games,negative,0.0,1.173003e+06
3,steam_games,recommendations,0.0,4.830455e+06
4,rawg_games,id,1.0,1.018370e+06
...,...,...,...,...
61,twitch_streamers,TOTAL_VIEWS,0.0,5.720000e+08
62,twitch_streamers,TOTAL_GAMES_STREAMED,1.0,2.980000e+03
63,twitch_streamers,ACTIVE_DAYS_PER_WEEK,0.0,7.000000e+00
64,steam_game_metadata,app_id,10.0,2.599300e+06


## NLP: Habr и YouTube

Для двух статей Habr выполняются очистка текста, токенизация, удаление частотных русских и английских служебных слов и TF-IDF-векторизация униграмм и биграмм. Для YouTube аналогично обрабатываются заголовки и описания видео.

YouTube-файл содержит текстовые метаданные, а не видеофайлы или аудиодорожки. Поэтому в этой работе векторизуются заголовок и описание; распознавание речи или анализ видеоряда не выполняются.

Две статьи Habr слишком малы для обобщения на всю индустрию. NLP здесь служит воспроизводимым описательным примером. Векторные матрицы и словари сохраняются для дальнейшего использования в 2.4 и 2.5.

In [8]:
from scipy.sparse import save_npz

RU_STOPWORDS = {
    "это", "как", "так", "его", "ее", "её", "что", "кто", "где",
    "когда", "или", "для", "при", "без", "над", "под", "про", "все",
    "всё", "они", "она", "оно", "мы", "вы", "они", "этот", "эта",
    "эти", "быть", "был", "была", "были", "есть", "из", "на", "по",
    "от", "до", "за", "не", "но", "а", "и", "в", "во", "с", "со",
    "the", "and", "for", "with", "from", "this", "that", "are", "was",
    "were", "you", "your", "have", "has", "into", "about",
}

def save_tfidf(frame, text_column, id_column, prefix):
    text_values = frame[text_column].fillna("").astype(str)
    valid = text_values.str.strip().ne("")
    frame = frame.loc[valid].copy()
    text_values = text_values.loc[valid].map(clean_text)

    if frame.empty:
        print(f"{prefix}: нет текстов для векторизации")
        return

    vectorizer = TfidfVectorizer(
        token_pattern=r"(?u)\b[а-яёa-z]{3,}\b",
        stop_words=sorted(RU_STOPWORDS),
        ngram_range=(1, 2),
        min_df=1,
        max_df=1.0,
        max_features=3000,
        sublinear_tf=True,
    )

    matrix = vectorizer.fit_transform(text_values)
    save_npz(TEXT / f"{prefix}_tfidf.npz", matrix)

    terms = vectorizer.get_feature_names_out()
    pd.DataFrame({
        "feature_index": range(len(terms)),
        "term": terms,
    }).to_csv(
        TEXT / f"{prefix}_tfidf_vocabulary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    frame[[id_column]].reset_index(drop=True).to_csv(
        TEXT / f"{prefix}_tfidf_documents.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"{prefix}: документов={matrix.shape[0]}, "
        f"признаков={matrix.shape[1]}, "
        f"ненулевых значений={matrix.nnz}"
    )

habr_for_nlp = pd.read_parquet(habr_output)
save_tfidf(
    habr_for_nlp,
    text_column="clean_text",
    id_column="url",
    prefix="habr",
)

youtube_df = pd.read_parquet(prepared["youtube_search_rows"])
youtube_df["video_text"] = (
    youtube_df.get("title", "").fillna("").astype(str)
    + " "
    + youtube_df.get("description", "").fillna("").astype(str)
)

# video_id может встречаться для нескольких поисковых запросов.
# В таблице документов оставляем по одной строке на видео.
youtube_text = youtube_df.drop_duplicates("video_id").copy()
youtube_text["video_text"] = youtube_text["video_text"].map(clean_text)

save_tfidf(
    youtube_text,
    text_column="video_text",
    id_column="video_id",
    prefix="youtube",
)

stage_times.append({
    "stage": "nlp",
    "seconds": time.perf_counter() - stage_start,
})
stage_start = time.perf_counter()

habr: документов=2, признаков=3000, ненулевых значений=3073
youtube: документов=298, признаков=3000, ненулевых значений=5856


## Сопоставление источников

Steam Recommendations и Steam Monthly Players связываются с каталогом Steam по Steam AppID. Эти значения относятся к одной системе идентификаторов.

Для RAWG, Wikidata и названий игр Twitch строятся кандидаты по нормализованному названию. Сохраняются исходные названия и идентификаторы, количество повторов и статус. Только уникальные совпадения обоих источников помечаются `unique_name_match_unverified`; неоднозначные сохраняются отдельно.

Поисковое название YouTube не трактуется как подтверждённая принадлежность видео к игре. Факты Twitch, YouTube и Steam с разной детализацией не сливаются в одну плоскую таблицу.

Повторы учитываются на уровне сущностей, а не строк наблюдений: строки одного Wikidata ID объединяются для подсчёта игр, а повторения Twitch по месяцам не считаются разными играми. Steam AppID и идентификаторы RAWG/Wikidata сохраняются строками, чтобы не терять формат ID.

In [9]:
mapping_start = time.perf_counter()

def normalize_name(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"[_\s]+", " ", value)
    return value.strip() or None


steam_names = pd.read_parquet(
    steam_output,
    columns=["AppID", "name"],
)
steam_names["AppID"] = steam_names["AppID"].astype("string")
steam_names["normalized_name"] = steam_names["name"].map(normalize_name)
steam_names = steam_names.dropna(subset=["normalized_name"])
steam_names = steam_names.drop_duplicates(["AppID", "normalized_name"])

steam_counts = steam_names.groupby("normalized_name")["AppID"].nunique()

rawg_names = pd.read_parquet(rawg_output, columns=["id", "name"])
wikidata_names = wikidata_df[["game", "gameLabel"]].copy()
twitch_games_df = pd.read_parquet(prepared["twitch_games"])

mapping_sources = {
    "RAWG": (rawg_names, "id", "name"),
    "Wikidata": (wikidata_names, "game", "gameLabel"),
    "Twitch": (
        twitch_games_df[["Game"]].rename(columns={"Game": "name"}),
        None,
        "name",
    ),
}

candidate_frames = []
mapping_frames = []

for source_name, (frame, id_column, name_column) in mapping_sources.items():
    source = frame.copy()
    source["normalized_name"] = source[name_column].map(normalize_name)
    source = source.dropna(subset=["normalized_name"])

    if id_column:
        source[id_column] = source[id_column].astype("string")
        source = source.drop_duplicates([id_column, "normalized_name"])
        source_counts = source.groupby("normalized_name")[id_column].nunique()
    else:
        # Twitch Games не имеет ID игры; оставляем одно наблюдение на название.
        source = source.drop_duplicates(["normalized_name"])
        source_counts = source["normalized_name"].value_counts()

    common_names = source_counts.index.intersection(steam_counts.index)
    candidates_for_source = pd.DataFrame({
        "normalized_name": common_names,
    })

    candidates_for_source["source"] = source_name
    candidates_for_source["source_count"] = (
        candidates_for_source["normalized_name"].map(source_counts)
    )
    candidates_for_source["steam_count"] = (
        candidates_for_source["normalized_name"].map(steam_counts)
    )

    is_unique = (
        candidates_for_source["source_count"].eq(1)
        & candidates_for_source["steam_count"].eq(1)
    )
    candidates_for_source["match_status"] = "ambiguous_name"
    candidates_for_source.loc[
        is_unique, "match_status"
    ] = "unique_name_match_unverified"

    candidate_frames.append(candidates_for_source)

    unique_names = candidates_for_source.loc[
        is_unique, ["normalized_name"]
    ]

    steam_unique = (
        steam_names
        .merge(unique_names, on="normalized_name", how="inner")
        [["AppID", "name", "normalized_name"]]
        .drop_duplicates("normalized_name")
        .rename(columns={
            "AppID": "steam_app_id",
            "name": "steam_name",
        })
    )

    source_unique = source.merge(
        unique_names,
        on="normalized_name",
        how="inner",
    ).drop_duplicates("normalized_name")

    source_unique = source_unique.rename(columns={
        name_column: "source_game_name",
    })

    if id_column:
        source_unique["source_game_id"] = (
            source_unique[id_column].astype("string")
        )
    else:
        source_unique["source_game_id"] = pd.Series(
            pd.NA,
            index=source_unique.index,
            dtype="string",
        )

    source_unique = source_unique[
        ["source_game_id", "source_game_name", "normalized_name"]
    ]

    matches = steam_unique.merge(
        source_unique,
        on="normalized_name",
        how="inner",
        validate="one_to_one",
    )
    matches["source"] = source_name
    matches["match_method"] = "exact_normalized_name"
    matches["match_status"] = "unique_name_match_unverified"

    mapping_frames.append(matches)

candidates = pd.concat(candidate_frames, ignore_index=True)
game_mapping = pd.concat(mapping_frames, ignore_index=True)

candidate_output = GAMES / "game_match_candidates.parquet"
mapping_output = GAMES / "game_source_mapping.parquet"

candidates.to_parquet(
    candidate_output,
    index=False,
    compression="zstd",
)
game_mapping.to_parquet(
    mapping_output,
    index=False,
    compression="zstd",
)

prepared["game_match_candidates"] = candidate_output
prepared["game_source_mapping"] = mapping_output

display(
    candidates.groupby(["source", "match_status"])
    .size()
    .rename("count")
    .reset_index()
)
display(game_mapping.head(10))

stage_times.append({
    "stage": "source_matching",
    "seconds": time.perf_counter() - mapping_start,
})

,source,match_status,count
0,RAWG,ambiguous_name,4358
1,RAWG,unique_name_match_unverified,102889
2,Twitch,ambiguous_name,33
3,Twitch,unique_name_match_unverified,1263
4,Wikidata,ambiguous_name,1
5,Wikidata,unique_name_match_unverified,3


,steam_app_id,steam_name,normalized_name,source_game_id,source_game_name,source,match_method,match_status
0,496350,Supipara - Chapter 1 Spring Has Come!,supipara chapter 1 spring has come,14508,Supipara - Chapter 1 Spring Has Come!,RAWG,exact_normalized_name,unique_name_match_unverified
1,1034400,Mystery Solitaire The Black Raven,mystery solitaire the black raven,308081,Mystery Solitaire The Black Raven,RAWG,exact_normalized_name,unique_name_match_unverified
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,버튜버 파라노이아 vtuber paranoia,991741,버튜버 파라노이아 - Vtuber Paranoia,RAWG,exact_normalized_name,unique_name_match_unverified
3,3631080,Maze Quest VR,maze quest vr,1002006,Maze Quest VR,RAWG,exact_normalized_name,unique_name_match_unverified
4,1654170,Agony VR,agony vr,951979,Agony VR,RAWG,exact_normalized_name,unique_name_match_unverified
5,1934300,Armored Brigade II,armored brigade ii,1001099,Armored Brigade II,RAWG,exact_normalized_name,unique_name_match_unverified
6,700340,Galacatraz: Eject Equip Escape,galacatraz eject equip escape,51647,Galacatraz: Eject Equip Escape,RAWG,exact_normalized_name,unique_name_match_unverified
7,1157670,Hepta Beats,hepta beats,600972,Hepta Beats,RAWG,exact_normalized_name,unique_name_match_unverified
8,1540330,MUMBA IV: Egypt Jewels,mumba iv egypt jewels,709029,MUMBA IV: Egypt Jewels ©,RAWG,exact_normalized_name,unique_name_match_unverified
9,1108640,OMNIMUS,omnimus,370174,OMNIMUS,RAWG,exact_normalized_name,unique_name_match_unverified


## Хранилище и повторный запуск

Parquet используется как переносимый колоночный формат подготовленных таблиц. DuckDB выбран как локальное аналитическое хранилище: он хранит раздельные факты и справочники и позволяет выполнять запросы без объединения источников с разной детализацией.

При повторном запуске подготовленные файлы и таблицы DuckDB пересоздаются из `raw/`. Связи между разными платформами сохраняются в отдельных таблицах кандидатов и соответствий. Время инвентаризации, подготовки, NLP, сопоставления и записи в хранилище сохраняется для дальнейшего сравнения в 2.5.

In [10]:
warehouse_start = time.perf_counter()
DATABASE = WAREHOUSE / "games.duckdb"
connection = duckdb.connect(str(DATABASE))

warehouse_rows = []

try:
    for table_name, parquet_path in prepared.items():
        parquet_sql = sql_literal(parquet_path)

        connection.execute(f"""
            CREATE OR REPLACE TABLE "{table_name}" AS
            SELECT * FROM read_parquet({parquet_sql})
        """)

        row_count = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]

        warehouse_rows.append({
            "table": table_name,
            "rows": row_count,
            "parquet": parquet_path.relative_to(ROOT).as_posix(),
        })

    # Месячные показатели Steam связываются с каталогом по AppID.
    connection.execute("""
        CREATE OR REPLACE VIEW v_steam_monthly_players AS
        SELECT
            p.*,
            g.name AS steam_catalog_name
        FROM steam_monthly_players AS p
        LEFT JOIN steam_games AS g
            ON CAST(p.steam_appid AS VARCHAR) = g.AppID
    """)

    # Рекомендации связываются с играми Steam по app_id.
    connection.execute("""
        CREATE OR REPLACE VIEW v_steam_recommendations AS
        SELECT
            r.*,
            g.title AS recommendation_game_title
        FROM steam_recommendations AS r
        LEFT JOIN steam_recommendation_games AS g
            ON r.app_id = g.app_id
    """)

    monthly_coverage = connection.execute("""
        SELECT
            COUNT(*) AS monthly_rows,
            COUNT(steam_catalog_name) AS rows_with_catalog_match
        FROM v_steam_monthly_players
    """).fetchdf()

finally:
    connection.close()

warehouse_summary = pd.DataFrame(warehouse_rows)
display(warehouse_summary)
display(monthly_coverage)

warehouse_summary.to_csv(
    QUALITY_DIR / "warehouse_summary_2_3.csv",
    index=False,
    encoding="utf-8-sig",
)

stage_times.append({
    "stage": "duckdb_warehouse",
    "seconds": time.perf_counter() - warehouse_start,
})

timing_df = pd.DataFrame(stage_times)
timing_df["recorded_at_utc"] = datetime.now(timezone.utc).isoformat()

timing_df.to_csv(
    QUALITY_DIR / "processing_times_2_3.csv",
    index=False,
    encoding="utf-8-sig",
)

print("DuckDB:", DATABASE.relative_to(ROOT))
display(timing_df)

,table,rows,parquet
0,steam_games,150466,processed/games/steam_games.parquet
1,rawg_games,899585,processed/games/rawg_games.parquet
2,steam_recommendation_games,50872,processed/recommendations/steam_recommendation...
3,steam_recommendations,41154794,processed/recommendations/steam_recommendation...
4,steam_users,14306064,processed/recommendations/steam_users.parquet
5,steam_monthly_players,612265,processed/players/steam_monthly_players.parquet
6,youtube_search_rows,299,processed/youtube/youtube_search_rows.parquet
7,twitch_games,21000,processed/twitch/twitch_games.parquet
8,twitch_global,105,processed/twitch/twitch_global.parquet
9,twitch_streamers,999,processed/twitch/twitch_streamers.parquet


,monthly_rows,rows_with_catalog_match
0,612265,578547


DuckDB: warehouse\games.duckdb


,stage,seconds,recorded_at_utc
0,raw_inventory,0.106320,2026-11-22T19:02:13.114749+00:00
1,prepare_steam_rawg,6.886736,2026-11-22T19:02:13.114749+00:00
2,prepare_csv_twitch_youtube_metadata,4.018172,2026-11-22T19:02:13.114749+00:00
3,prepare_wikidata_habr,0.524157,2026-11-22T19:02:13.114749+00:00
4,nlp,1.569218,2026-11-22T19:02:13.114749+00:00
5,source_matching,4.654155,2026-11-22T19:02:13.114749+00:00
6,duckdb_warehouse,3.784675,2026-11-22T19:02:13.114749+00:00


## Результат и ограничения

В `processed/` созданы таблицы Steam Games, RAWG, Steam Recommendations и пользователей, Steam Monthly Players, Twitch, YouTube, Habr и Wikidata. Качество таблиц, пропуски и примеры строк сохранены в `docs/data_quality/`. DuckDB содержит подготовленные таблицы и таблицы сопоставлений. Для текстов Habr и метаданных YouTube сохранены TF-IDF-матрицы и словари.

Совпадения по названию не являются ручной проверкой идентичности игр. YouTube связан с поисковым запросом, а не доказанной игрой в видео. Видеофайлы и аудио не загружались; анализируется только текст YouTube-метаданных. Две статьи Habr позволяют показать обработку NLP, но не дают репрезентативной оценки всей отрасли.

В 2.4 для анализа используются подготовленные таблицы и векторные признаки. В 2.5 можно сопоставить сохранённое время этапов с альтернативными способами загрузки, подготовки, сопоставления и анализа. В 2.6 следует указывать фактически использованные таблицы, поля, ограничения и результаты, а не считать все записи подтверждённо объединёнными.